# Stereo Vision

The input is two images, left and right, and the goal is to compute the disparity using two ways: **Block Matching** and **Dynamic Programming**. 

In [1]:
import numpy as np

## Block Matching
- This is done by matching each pixel in the left image to a pixel in the right image. 
- Since there is no rectification needed, the row in the left image will only be matched with its equivalent in the right image. 
- The disparity will be calculated in two ways, once using the cost as the Sum of Absolute - Differences (SAD) and another time using Sum of Squared Differences.
- This will be done for windows of size w where w = 1, 5 and 9.
- 6 maps will be produced: 2 maps for each window size, once using SAD and the other using SSD.

In [2]:
def calculate_window_cost(left_w, right_w, use_sad=True):
    if use_sad:
        return np.sum(np.abs(left_w - right_w)) # Sum of Absolute Differences
    else:
        return np.sum((left_w - right_w) ** 2) # Sum of Squared Differences

In [3]:
def compute_disparity(left_image, right_image, window_size, max_disparity, use_sad=True):
    
    height, width = left_image.shape
    
    # disparity value for each pixel
    disparity_map = np.zeros((height, width), dtype=np.uint8)
    
    # get half window size to avoid going out of bounds
    half_window = window_size // 2

    # traverse vertically
    for y in range(half_window, height - half_window): 
        # traverse horizontally
        for x in range(half_window, width - half_window):
            
            # get window from left image centered at the (x,y) current pixel of size w x w
            window_left = left_image[y - half_window : y + half_window + 1, x - half_window : x + half_window + 1]
            
            min_cost = float('inf')
            best_d = 0
            
            # get candidate disparities
            for d in range(max_disparity + 1):
                
                # don't go out of bounds
                if x - half_window - d >= 0:
                    
                    # get window from right image centered at the (x-d,y) current pixel of size w x w
                    window_right = right_image[y - half_window : y + half_window + 1, x - half_window - d : x + half_window + 1 - d]
                    
                    # the lowest window cost will give the best match from which we can get the disparity
                    cost = calculate_window_cost(window_left, window_right, use_sad)
                    
                    if cost < min_cost:
                        min_cost = cost
                        best_d = d
                        
            # save best disparity for that pixel
            disparity_map[y, x] = best_d
            
    return disparity_map

In [ ]:
window_sizes = [1, 5, 9]

